# 07.3 String Immutability

A string can never be changed. Every method that looks like it modifies a string
actually **returns a new one**.

This is the single most common source of "why didn't my code work?" for people
new to Python — and once you see it, it never catches you again.

## The one rule

```python
text.upper()          # computes a new string, then throws it away
text = text.upper()   # computes a new string and KEEPS it
```

## Easy — Seeing that strings do not change

Four short examples that make the rule obvious.

In [ ]:
# EXAMPLE 1: You cannot change one character
# Trying to assign to a position raises an error.
word = "Python"

try:
    word[0] = "J"
except TypeError as error:
    print("TypeError:", error)

In [ ]:
# EXAMPLE 2: Methods return a NEW string
# upper() gives back a new string. The original is untouched.
word = "python"

louder = word.upper()

print("original:", word)
print("result:  ", louder)

In [ ]:
# EXAMPLE 3: The classic mistake
# Calling a method without keeping the result does nothing useful.
word = "python"

word.upper()

print("After calling word.upper():", word)
print("The new string was created, then discarded.")

In [ ]:
# EXAMPLE 4: The fix - assign the result
# Capture what the method returns.
word = "python"

word = word.upper()

print("After word = word.upper():", word)

In [ ]:
# EXAMPLE 5: Every string method behaves this way
# strip, replace and lower all return new strings.
text = "  Hello World  "

print("original:      ", repr(text))
print("after strip(): ", repr(text.strip()))
print("original again:", repr(text), "<- still unchanged")

In [ ]:
# EXAMPLE 6: Building up a string with +=
# += on a string creates a new string each time - it does not modify.
result = ""
result += "a"
result += "b"
result += "c"

print(result)

## Medium — Why immutability matters

The practical consequences you will actually meet.

In [ ]:
# EXAMPLE 7: Watching the identity change
# id() reveals that a NEW object is created each time.
word = "python"
first_id = id(word)

word = word.upper()
second_id = id(word)

print("id before:", first_id)
print("id after: ", second_id)
print("Same object?", first_id == second_id)

In [ ]:
# EXAMPLE 8: Compare with a list, which IS mutable
# A list method changes the object in place - the id stays the same.
items = ["b", "a"]
before = id(items)

items.sort()

print("list after sort():", items)
print("Same object?", id(items) == before)
print("")
print("A list changed in place. A string never can.")

In [ ]:
# EXAMPLE 9: Two names for one string are always safe
# Because nothing can modify a string, sharing it is risk-free.
original = "hello"
alias = original

original = original.upper()

print("original:", original)
print("alias:   ", alias, "<- unaffected")

In [ ]:
# EXAMPLE 10: Strings can be dictionary keys
# Immutable means the hash never changes, so strings work as keys.
prices = {"apple": 10, "banana": 5}

print(prices["apple"])
print("")
print("A list cannot be a key, because it could change:")
try:
    {["apple"]: 10}
except TypeError as error:
    print("TypeError:", error)

In [ ]:
# EXAMPLE 11: Changing one character - the real way
# Build a new string from the pieces you want.
word = "Python"

changed = "J" + word[1:]

print("original:", word)
print("changed: ", changed)

In [ ]:
# EXAMPLE 12: Changing a character in the middle
# Slice around the position you want to replace.
word = "Python"
position = 2

changed = word[:position] + "T" + word[position + 1:]

print("original:", word)
print("changed: ", changed)

In [ ]:
# EXAMPLE 13: Using a list when you need many edits
# Convert to a list, edit freely, then join back.
word = "Python"

characters = list(word)
characters[0] = "J"
characters[-1] = "!"
rebuilt = "".join(characters)

print("original:", word)
print("rebuilt: ", rebuilt)

## Hard — Performance and interning

What immutability costs, and what it buys.

In [ ]:
# EXAMPLE 14: Why += in a loop is slow
# Each += builds a whole new string, copying everything so far.
import time

def with_plus(count):
    """Build a string by repeated concatenation."""
    result = ""
    for number in range(count):
        result += str(number)
    return result


def with_join(count):
    """Collect the parts and join once at the end."""
    parts = []
    for number in range(count):
        parts.append(str(number))
    return "".join(parts)


size = 20000

start = time.perf_counter()
with_plus(size)
plus_time = (time.perf_counter() - start) * 1000

start = time.perf_counter()
with_join(size)
join_time = (time.perf_counter() - start) * 1000

print(f"repeated +=: {plus_time:.1f} ms")
print(f"''.join():   {join_time:.1f} ms")

In [ ]:
# EXAMPLE 15: The join pattern, written properly
# This is the idiom to reach for whenever you build text in a loop.
words = ["Python", "is", "readable"]

# Collect parts, then join once.
sentence = " ".join(words)

print(sentence)

In [ ]:
# EXAMPLE 16: Joining with a generator
# You do not even need an intermediate list.
numbers = range(5)

joined = ", ".join(str(number) for number in numbers)

print(joined)

In [ ]:
# EXAMPLE 17: Interning makes repeated strings cheap
# Python reuses one object for identical simple literals.
first = "hello"
second = "hello"

print("Same object?", first is second)
print("id first: ", id(first))
print("id second:", id(second))
print("")
print("This is only safe BECAUSE strings cannot change.")

In [ ]:
# EXAMPLE 18: Strings built at runtime are separate objects
# Interning applies to literals, not to computed strings.
literal = "hello"
computed = "".join(["h", "e", "l", "l", "o"])

print("Equal in value?", computed == literal)
print("Same object?   ", computed is literal)
print("")
print("Always compare strings with ==, never with `is`.")

In [ ]:
# EXAMPLE 19: sys.intern forces sharing
# You can ask Python to intern a computed string.
import sys

computed = "".join(["h", "i", "!"])
interned = sys.intern(computed)
other = sys.intern("hi!")

print("Same object after interning?", interned is other)
print("")
print("Useful when comparing millions of repeated strings,")
print("because identity comparison is faster than character comparison.")

## Takeaways

1. Strings are **immutable** — `text[0] = "x"` raises `TypeError`.
2. Every string method **returns a new string**. `text.upper()` alone does
   nothing; you must write `text = text.upper()`.
3. To change one character, rebuild with slices:
   `word[:i] + new + word[i+1:]`.
4. For many edits, convert to a list, edit, then `"".join()`.
5. Repeated `+=` in a loop is **slow** — it copies everything each time. Use
   `"".join()`.
6. Immutability is what lets strings be **dictionary keys** and be safely
   **shared** and **interned**.
7. Compare strings with `==`, never `is` — interning is not guaranteed.

## Try it yourself

1. Try to change a character by index. Read the error.
2. Call `.upper()` without assigning, then print. Explain what happened.
3. Replace the third character of a word using slices.
4. Build a 10,000-character string with `+=` and with `join`. Time both.
5. Check whether two identical literals share an object with `is`.